# Project Update 1 — Data Engineering for Streaming Code-Switching

Pipeline: **SwitchLingua text (single- and multi-turn) → pair / script filtering → balanced train/val/test split → word-level LID → predictive labels (y_sw, y_dur) → sub-word alignment → Streaming DataLoader** yielding context windows `[x_{t-N}, …, x_t]`.

Label rules (from the project spec):
- `y_sw(t) = 1` if `LID(x_{t+1}) ≠ LID(x_t)`, else `0`. The last token has no next token → ignored (`-100`).
- `y_dur(t)` = length bin of the new-language segment starting at `x_{t+1}` when `y_sw(t)=1`: **0** = 1–2 tokens, **1** = 3–6, **2** = 7+. Otherwise `-1` (ignored).

In [1]:
# !pip install datasets transformers torch pandas wordfreq lingua-language-detector
import ast, random, re, unicodedata
import numpy as np, torch, pandas as pd
from collections import Counter, defaultdict
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import IterableDataset, DataLoader, get_worker_info
from wordfreq import zipf_frequency

## 1. Load SwitchLingua and inspect the schema
The dataset is gated: accept its conditions on the Hugging Face page and log in first (`huggingface-cli login` or `from huggingface_hub import login; login()`).

It ships a single `train` split, **sorted by language pair** (Arabic rows 0–8547, Cantonese 8548–127795, …).
Taking the first *k* rows therefore only ever sees one or two pairs; Section 4 builds our own balanced, shuffled splits.

In [2]:
ds = load_dataset("Shelton1013/SwitchLingua_text")["train"]   # the only split provided
ds

Dataset({
    features: ['topic', 'tense', 'perspective', 'cs_ratio', 'gender', 'age', 'education_level', 'first_language', 'second_language', 'conversation_type', 'cs_function', 'cs_type', 'data_generation_result', 'fluency_result', 'naturalness_result', 'cs_ratio_result', 'social_cultural_result', 'summary', 'score'],
    num_rows: 234172
})

## 2. Configuration
The dataset has **no token-level language labels**: each row gives `first_language`, `second_language` and the sentence in `data_generation_result`. Word-level LIDs are therefore derived in Section 3.

`SELECTED_PAIRS` should match the language pairs chosen in Part 1.
Languages written in a different script from English (Arabic, Hindi, Cantonese, …) can be labeled by script.
Latin-script languages (Spanish, German) are labeled with the frequency-based LID in Section 3.

In [3]:
TEXT_COL = "data_generation_result"
# Six typologically diverse pairs:
#   ar (Semitic, RTL) | hi (Indo-Aryan, SOV) | yue (Sinitic, low-resource) | es (Romance)
#   ko (Koreanic, agglutinative, SOV) | de (Germanic, closest to English)
SELECTED_PAIRS = [("en", "ar"), ("en", "hi"), ("en", "yue"),
                  ("en", "es"), ("en", "ko"), ("en", "de")]
PAIR_NAMES = ["-".join(p) for p in SELECTED_PAIRS]
PAIR_ID = {p: i for i, p in enumerate(SELECTED_PAIRS)}

CONV_TYPES = ["single_turn", "multi-turn"]
ROWS_PER_PAIR_TYPE = 4000  # balance pairs: Cantonese has ~60k rows per conversation type, the others ~4k
SPLIT_FRAC = (0.8, 0.1, 0.1)
SEED = 0
WINDOW = 64                # N: context window length (sub-words)
MODEL_NAME = "xlm-roberta-base"

LANG_CODE = {"English": "en", "Arabic": "ar", "Hindi": "hi", "Cantonese": "yue", "Chinese": "zh",
             "Japanese": "ja", "Korean": "ko", "Russian": "ru", "Spanish": "es", "French": "fr",
             "German": "de", "Italian": "it", "Malay": "ms", "Vietnamese": "vi", "Philippines": "tl"}

def pair_of(first, second):
    # ("en", X) for English–X rows; None for rows without English (e.g. German–French)
    a, b = LANG_CODE.get(first), LANG_CODE.get(second)
    if a == "en" and b not in (None, "en"): return ("en", b)
    if b == "en" and a not in (None, "en"): return ("en", a)
    return None

def f1(y, yhat):
    # F1 of the positive (switch) class
    y, yhat = np.asarray(y), np.asarray(yhat)
    tp = int(((y == 1) & (yhat == 1)).sum()); fp = int(((y == 0) & (yhat == 1)).sum()); fn = int(((y == 1) & (yhat == 0)).sum())
    return 2 * tp / (2 * tp + fp + fn) if tp else 0.0

def get_pair(ex):
    return pair_of(ex["first_language"], ex["second_language"])

def get_turns(ex):
    # data_generation_result is a stringified list: sentences of one speaker (single_turn)
    # or alternating speaker turns (multi-turn). Returns one string per speaker turn.
    raw = ex[TEXT_COL]
    if isinstance(raw, str) and raw.strip().startswith("["):
        try:
            raw = ast.literal_eval(raw)
        except (ValueError, SyntaxError):
            pass
    parts = list(map(str, raw)) if isinstance(raw, list) else [str(raw)]
    return parts if ex["conversation_type"] == "multi-turn" else [" ".join(parts)]

def get_text(ex):
    return " ".join(get_turns(ex))

## 3. Word-level LID and predictive label generation
Text is pre-tokenized into words (each Han character counts as one token, since Chinese has no spaces).
Language-neutral tokens (punctuation, digits) inherit the previous token's language.

- **Script-based LID** (ar, hi, yue, ko): the Unicode script of a word decides its language.
- **Frequency-based LID** (es, de): compare the word's Zipf frequency in English vs. the other language (`wordfreq`).
  If both scores are low (unknown word) or too close (`die`, `in`, `no`), the word inherits the previous word's language.

In [4]:
SCRIPT_OF_LANG = {
    "en": "LATIN", "ar": "ARABIC", "hi": "DEVANAGARI", "yue": "HAN", "zh": "HAN",
    "ja": "JAPANESE", "ko": "HANGUL", "ru": "CYRILLIC",
    "es": "LATIN", "fr": "LATIN", "de": "LATIN", "it": "LATIN",
}

def char_script(ch):
    try:
        name = unicodedata.name(ch)
    except ValueError:
        return None
    for key in ["LATIN", "ARABIC", "DEVANAGARI", "CYRILLIC", "HANGUL"]:
        if name.startswith(key):
            return key
    if name.startswith("CJK"):
        return "HAN"
    if name.startswith(("HIRAGANA", "KATAKANA")):
        return "JAPANESE"
    return None

TOKEN_RE = re.compile(r"[\u4e00-\u9fff\u3400-\u4dbf]|[^\s\u4e00-\u9fff\u3400-\u4dbf]+")

def pre_tokenize(text):
    return TOKEN_RE.findall(text)

def word_lid(word, pair, latin_lid_fn=None):
    scripts = [char_script(c) for c in word if c.isalpha()]
    scripts = [s for s in scripts if s]
    if not scripts:
        return None
    script = Counter(scripts).most_common(1)[0][0]   # deterministic tie-break (first seen)
    l1, l2 = pair
    s1, s2 = SCRIPT_OF_LANG[l1], SCRIPT_OF_LANG[l2]
    if s1 != s2:
        if script == "HAN" and "JAPANESE" in (s1, s2):
            script = "JAPANESE"          # kanji inside a Japanese-English pair
        if script == s1: return l1
        if script == s2: return l2
        return None
    if latin_lid_fn is None:
        raise ValueError(f"{pair} share a script; provide latin_lid_fn")
    return latin_lid_fn(word, pair)

def fill_neutral(lids):
    out, prev = [], None
    for l in lids:
        if l is None: l = prev
        out.append(l); prev = l
    first = next((l for l in out if l is not None), None)
    return [l if l is not None else first for l in out]

def dur_bin(n):
    return 0 if n <= 2 else (1 if n <= 6 else 2)

def make_labels(lids):
    T = len(lids)
    run_len = [0] * T
    for i in range(T - 1, -1, -1):
        run_len[i] = 1 + (run_len[i + 1] if i + 1 < T and lids[i + 1] == lids[i] else 0)
    y_sw, y_dur = [], []
    for t in range(T):
        if t == T - 1:
            y_sw.append(-100); y_dur.append(-1); continue
        sw = int(lids[t + 1] != lids[t])
        y_sw.append(sw)
        y_dur.append(dur_bin(run_len[t + 1]) if sw else -1)
    return y_sw, y_dur

# ---------- Frequency-based LID for Latin-script pairs (e.g. en-es, en-de) ----------
WORD_CHARS = re.compile(r"[^\w'-]")

def make_latin_lid(freq_fn, margin=1.0, min_zipf=1.5):
    # freq_fn(word, lang) -> Zipf frequency (wordfreq.zipf_frequency).
    # Returns None when a word is unknown in both languages or the scores are too
    # close (e.g. 'die', 'no', 'in'); fill_neutral() then uses the previous word's language.
    def latin_lid(word, pair):
        w = WORD_CHARS.sub("", word)
        if not w or w.isdigit():
            return None
        l1, l2 = pair
        f1, f2 = freq_fn(w, l1), freq_fn(w, l2)
        if max(f1, f2) < min_zipf or abs(f1 - f2) < margin:
            return None
        return l1 if f1 > f2 else l2
    return latin_lid

In [5]:
LATIN_LID = make_latin_lid(zipf_frequency, margin=1.0, min_zipf=1.5)

### Sanity check on toy sentences

In [6]:
def show(sentence, pair):
    w = pre_tokenize(sentence)
    raw = [word_lid(x, pair, LATIN_LID) for x in w]
    l = fill_neutral(raw)
    sw, dur = make_labels(l)
    return pd.DataFrame({"token": w, "raw LID": raw, "LID": l, "y_sw": sw, "y_dur": dur})

show("我今日好 busy, 因為 I have three meetings 啊", ("en", "yue"))

,token,raw LID,LID,y_sw,y_dur
0,我,yue,yue,0,-1
1,今,yue,yue,0,-1
2,日,yue,yue,0,-1
3,好,yue,yue,1,0
4,"busy,",en,en,1,0
5,因,yue,yue,0,-1
6,為,yue,yue,1,1
7,I,en,en,0,-1
8,have,en,en,0,-1
9,three,en,en,0,-1


In [7]:
show("In Zukunft wird er die neuesten Technologien nutzen, um more innovative solutions zu finden.", ("en", "de"))

,token,raw LID,LID,y_sw,y_dur
0,In,NaN,de,0,-1
1,Zukunft,de,de,0,-1
2,wird,de,de,0,-1
3,er,de,de,0,-1
4,die,de,de,0,-1
5,neuesten,de,de,0,-1
6,Technologien,de,de,0,-1
7,"nutzen,",de,de,0,-1
8,um,de,de,1,1
9,more,en,en,0,-1


In [8]:
show("Mañana tengo que ir al office porque tenemos un meeting muy importante.", ("en", "es"))

,token,raw LID,LID,y_sw,y_dur
0,Mañana,es,es,0,-1
1,tengo,es,es,0,-1
2,que,es,es,0,-1
3,ir,es,es,0,-1
4,al,es,es,1,0
5,office,en,en,1,1
6,porque,es,es,0,-1
7,tenemos,es,es,0,-1
8,un,es,es,1,0
9,meeting,en,en,1,0


## 4. Row selection and train / val / test split
Filters, in order:
1. **English–X pairs only.** `German–French` rows contain no English and are excluded (the old `get_pair` could leak them into en-de).
2. **Both conversation types** (`CONV_TYPES`). Multi-turn rows are kept as dialogues; see Section 6 for how turns are handled.
3. **Script check** for script-based pairs: the row must contain at least one character in X's script.
   This removes **romanized Hindi (Hinglish)**, ~58% of single-turn Hindi rows, which script-based LID would label entirely as English.
4. **Balance**: sample at most `ROWS_PER_PAIR_TYPE` rows per pair × conversation type.

The split is done at the **row (sentence / dialogue) level**, stratified by pair × conversation type × `cs_type`, so prefixes of one row never appear in two splits.
Row order is shuffled so that pairs are interleaved when streamed.

In [9]:
SCRIPT_RE = {"ar": re.compile(r"[\u0600-\u06ff]"), "hi": re.compile(r"[\u0900-\u097f]"),
             "yue": re.compile(r"[\u4e00-\u9fff\u3400-\u4dbf]"), "zh": re.compile(r"[\u4e00-\u9fff\u3400-\u4dbf]"),
             "ko": re.compile(r"[\uac00-\ud7af\u1100-\u11ff]"), "ru": re.compile(r"[\u0400-\u04ff]"),
             "ja": re.compile(r"[\u3040-\u30ff]")}

def has_l2_script(text, pair):
    rx = SCRIPT_RE.get(pair[1])
    return True if rx is None else bool(rx.search(text))

meta = ds.select_columns(["first_language", "second_language", "conversation_type", "cs_type", TEXT_COL]).to_pandas()
pairs = [pair_of(a, b) for a, b in zip(meta.first_language, meta.second_language)]
meta["pair"] = ["-".join(p) if p else None for p in pairs]
meta["is_single"] = meta.conversation_type == "single_turn"
meta["script_ok"] = [p is None or has_l2_script(t, p) for p, t in zip(pairs, meta[TEXT_COL])]

sel = meta[meta.pair.isin(PAIR_NAMES) & meta.conversation_type.isin(CONV_TYPES)]
funnel = pd.DataFrame({
    "all rows": sel.groupby("pair").size(),
    "single (script ok)": sel[sel.is_single & sel.script_ok].groupby("pair").size(),
    "multi (script ok)": sel[~sel.is_single & sel.script_ok].groupby("pair").size(),
}).reindex(PAIR_NAMES)
sel = sel[sel.script_ok]

rng = np.random.default_rng(SEED)
split_idx = {"train": [], "val": [], "test": []}
for _, g in sel.groupby(["pair", "conversation_type"]):
    g = g.sample(n=min(ROWS_PER_PAIR_TYPE, len(g)), random_state=SEED)
    for _, gc in g.groupby("cs_type"):          # stratify by cs_type within each pair × conversation type
        a, b = int(len(gc) * SPLIT_FRAC[0]), int(len(gc) * (SPLIT_FRAC[0] + SPLIT_FRAC[1]))
        split_idx["train"] += gc.index[:a].tolist()
        split_idx["val"] += gc.index[a:b].tolist()
        split_idx["test"] += gc.index[b:].tolist()
splits = {k: ds.select(rng.permutation(v).tolist()) for k, v in split_idx.items()}

for k, v in split_idx.items():
    funnel[k] = meta.loc[v].groupby("pair").size()
funnel

,all rows,single (script ok),multi (script ok),train,val,test
pair,,,,,,
en-ar,8548,4251,4297,6398,800,802
en-hi,8381,1767,1632,2716,341,342
en-yue,119248,59987,59260,6397,799,804
en-es,8110,4052,4058,6398,800,802
en-ko,8266,4181,4085,6397,800,803
en-de,6819,3414,3405,5453,681,685


### Data checks
1. **Hindi script**: share of Hindi rows that contain Devanagari, per conversation type (the rest are romanized and were dropped above).
2. **Latin-pair LID quality** (es/de) is estimated automatically below, without manual annotation.

In [10]:
hi = meta[meta.pair == "en-hi"]
hi.groupby("conversation_type").script_ok.agg(rows="size", devanagari_share="mean", kept="sum")

,rows,devanagari_share,kept
conversation_type,,,
multi-turn,4151,0.393158,1632
single_turn,4230,0.417730,1767


### Automatic evaluation of the Latin-pair LID
The frequency-based LID is compared against independent reference systems on `N_EVAL_ROWS` validation rows per Latin pair.
Only words containing letters and labeled with a language by both systems are compared.

| Reference | Pairs | What it measures |
|---|---|---|
| **LinCE model** (`sagorsarker/codeswitch-spaeng-lid-lince`, BERT trained on the LinCE Spanish–English word-level LID benchmark) | en-es | closest to real accuracy; its `ne` / `other` / `mixed` / `ambiguous` words are skipped |
| **lingua** `detect_multiple_languages_of` (n-gram mixed-language detector) | en-es, en-de | agreement with a second, independent annotator; tends to miss 1–2-word insertions |
| **Sentence-level silver labels**: sentences that lingua finds monolingual (one span, confidence ≥ 0.95, ≥ 4 words) give every word the sentence's language | en-es, en-de | errors on easy, whole-sentence switches only (an upper bound on accuracy) |

Besides word-level agreement, the switch labels derived from each reference are compared with ours (F1 of `y_sw = 1`), since those are what the model learns.
Disagreements are written to `lid_disagreements.csv` for inspection.

In [11]:
from lingua import Language, LanguageDetectorBuilder
from transformers import AutoModelForTokenClassification

N_EVAL_ROWS = 300
LATIN_PAIRS = [p for p in SELECTED_PAIRS if SCRIPT_OF_LANG[p[0]] == SCRIPT_OF_LANG[p[1]]]
LINGUA_LANG = {"en": Language.ENGLISH, "es": Language.SPANISH, "de": Language.GERMAN}
LINGUA_CODE = {v: k for k, v in LINGUA_LANG.items()}
DETECTORS = {p: LanguageDetectorBuilder.from_languages(*(LINGUA_LANG[l] for l in p)).build() for p in LATIN_PAIRS}

LINCE_NAME = "sagorsarker/codeswitch-spaeng-lid-lince"
LINCE_MAP = {"en": "en", "spa": "es"}     # every other LinCE label -> None (skipped)
lince_tok = AutoTokenizer.from_pretrained(LINCE_NAME)
lince = AutoModelForTokenClassification.from_pretrained(LINCE_NAME).eval()

def lince_lid(words):
    enc = lince_tok(words, is_split_into_words=True, truncation=True, max_length=512, return_tensors="pt")
    with torch.no_grad():
        pred = lince(**enc).logits.argmax(-1)[0].tolist()
    out, seen = [None] * len(words), set()
    for i, w in enumerate(enc.word_ids()):
        if w is not None and w not in seen:       # label of the first sub-word
            seen.add(w)
            out[w] = LINCE_MAP.get(lince.config.id2label[pred[i]])
    return out

def span_lid(starts, spans):
    # language of the span containing each word's start offset
    return [next((l for a, b, l in spans if a <= s < b), None) for s in starts]

def lingua_lid(text, starts, pair):
    spans = [(r.start_index, r.end_index, LINGUA_CODE[r.language])
             for r in DETECTORS[pair].detect_multiple_languages_of(text)]
    return span_lid(starts, spans)

SENT_RE = re.compile(r"[^.!?]+[.!?]*")

def sentence_lid(text, pair, min_conf=0.95, min_words=4):
    det, spans = DETECTORS[pair], []
    for m in SENT_RE.finditer(text):
        sent = m.group()
        if len(pre_tokenize(sent)) < min_words or len(det.detect_multiple_languages_of(sent)) != 1:
            continue
        top = det.compute_language_confidence_values(sent)[0]
        if top.value >= min_conf:
            spans.append((m.start(), m.end(), LINGUA_CODE[top.language]))
    return spans

W1016 14:30:34.404000 28610 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: sagorsarker/codeswitch-spaeng-lid-lince
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
def ref_switches(ref):
    # y_sw implied by a reference labeling (skipped words inherit the previous language)
    return make_labels(fill_neutral(ref))[0]

val_meta = meta.loc[split_idx["val"]]
agree = defaultdict(lambda: [0, 0])        # (pair, ref) -> [agree, compared]
sw_pairs = defaultdict(lambda: ([], []))   # (pair, ref) -> (ours, reference) y_sw
disagreements = []
for pair in LATIN_PAIRS:
    pname = "-".join(pair)
    idx = val_meta.index[val_meta.pair == pname].tolist()
    for i in random.Random(SEED).sample(idx, min(N_EVAL_ROWS, len(idx))):
        for text in get_turns(ds[i]):
            matches = list(TOKEN_RE.finditer(text))
            if len(matches) < 2:
                continue
            words, starts = [m.group() for m in matches], [m.start() for m in matches]
            ours = fill_neutral([word_lid(w, pair, LATIN_LID) for w in words])
            refs = {"lingua": lingua_lid(text, starts, pair),
                    "sentence-level": span_lid(starts, sentence_lid(text, pair))}
            if pair == ("en", "es"):
                refs["LinCE model"] = lince_lid(words)
            for name, ref in refs.items():
                for w, o, r in zip(words, ours, ref):
                    if r is None or not any(c.isalpha() for c in w):
                        continue
                    agree[pname, name][0] += o == r
                    agree[pname, name][1] += 1
                    if o != r:
                        disagreements.append({"pair": pname, "reference": name, "word": w, "ours": o, "ref": r, "text": text})
                if name != "sentence-level":
                    sw_pairs[pname, name][0].extend(make_labels(ours)[0][:-1])
                    sw_pairs[pname, name][1].extend(ref_switches(ref)[:-1])

lid_eval = pd.DataFrame([{"pair": p, "reference": r, "words compared": n, "word agreement": round(a / n, 3),
                          "y_sw F1 vs reference": (round(f1(np.array(sw_pairs[p, r][1]), np.array(sw_pairs[p, r][0])), 3)
                                                   if (p, r) in sw_pairs else None)}
                         for (p, r), (a, n) in agree.items()])
pd.DataFrame(disagreements).to_csv("lid_disagreements.csv", index=False)
lid_eval

,pair,reference,words compared,word agreement,y_sw F1 vs reference
0,en-es,lingua,15870,0.949,0.631
1,en-es,LinCE model,15287,0.993,0.858
2,en-es,sentence-level,9229,0.961,NaN
3,en-de,lingua,14858,0.951,0.629
4,en-de,sentence-level,9303,0.959,NaN


In [13]:
# Most frequent disagreements: which words does the frequency-based LID get "wrong"?
dis = pd.DataFrame(disagreements)
(dis[dis.reference != "sentence-level"]
 .assign(word=lambda d: d.word.str.lower().str.strip(".,!?;:\"'()"))
 .groupby(["pair", "reference", "word", "ours", "ref"]).size().rename("count")
 .sort_values(ascending=False).head(20))

pair   reference    word      ours  ref
en-es  lingua       right     en    es     66
en-de  lingua       right     en    de     57
en-es  lingua       y         es    en     54
                    you       en    es     47
en-de  lingua       i         en    de     46
                    you       en    de     44
en-es  lingua       know      en    es     43
en-de  lingua       know      en    de     42
                    hat       de    en     27
en-es  lingua       a         en    es     26
en-de  lingua       honestly  en    de     26
en-es  lingua       honestly  en    es     23
en-de  lingua       es        de    en     16
en-es  LinCE model  wow       es    en     16
en-de  lingua       anyway    en    de     15
en-es  lingua       anyway    en    es     14
en-de  lingua       in        en    de     12
en-es  lingua       me        es    en     12
                    ha        es    en     11
                    it's      en    es     10
Name: count, dtype: int64

## 5. Sub-word alignment
Each word's language ID is copied to all its sub-words. The word-level labels are placed on the **last sub-word** of each word,
because that is the position from which the model predicts the next word. Other sub-words are ignored (`-100` / `-1`).

In [14]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def align_to_subwords(words, lids, y_sw, y_dur, max_len=512):
    enc = tokenizer(words, is_split_into_words=True, add_special_tokens=False,
                    truncation=True, max_length=max_len)
    wids, ids = enc.word_ids(), enc["input_ids"]
    sw, dur, lid = [-100] * len(ids), [-1] * len(ids), [None] * len(ids)
    for i, w in enumerate(wids):
        if w is None:
            continue
        lid[i] = lids[w]
        nxt = wids[i + 1] if i + 1 < len(wids) else None
        # last sub-word of a word that was not cut off by truncation
        is_last = (nxt is not None and nxt != w) or (nxt is None and w == len(words) - 1)
        if is_last:
            sw[i], dur[i] = y_sw[w], y_dur[w]
    return ids, sw, dur, lid

words = pre_tokenize("我今日好 busy, 因為 I have three meetings 啊")
lids = fill_neutral([word_lid(w, ("en", "yue")) for w in words])
ids, sw, dur, lid = align_to_subwords(words, lids, *make_labels(lids))
pd.DataFrame({"subword": tokenizer.convert_ids_to_tokens(ids), "LID": lid, "y_sw": sw, "y_dur": dur})

,subword,LID,y_sw,y_dur
0,▁我,yue,0,-1
1,▁今,yue,0,-1
2,▁,yue,-100,-1
3,日,yue,0,-1
4,▁,yue,-100,-1
5,好,yue,1,0
6,▁busy,en,-100,-1
7,",",en,1,0
8,▁,yue,-100,-1
9,因,yue,0,-1


## 6. Streaming DataLoader
For every labeled position `t`, the dataset yields the prefix window `[x_{t-N+1}, …, x_t]` and the labels for `x_{t+1}`.
The model only ever sees the past. Windows are **left-padded** so the prediction position is always the last one.

**Multi-turn dialogues**: each turn is labeled on its own and the turns are joined with the separator `</s>`, so the model sees the
previous turns as context (still prefix-only). Labels at the last word of a turn are ignored: the next word belongs to the other speaker,
so a language change there is a speaker change rather than a code-switch, and bursts never extend across turns.

Each window also carries `pair_id` (for the per-pair F1) and `multi_turn`.

In [15]:
def process_example(ex, latin_lid_fn=LATIN_LID, max_len=512):
    # Each speaker turn is labeled separately (its last word gets -100), then turns are joined with </s>.
    pair = get_pair(ex)
    ids, sw, dur, lid = [], [], [], []
    for turn in get_turns(ex):
        words = pre_tokenize(turn)
        if not words:
            continue
        lids = fill_neutral([word_lid(w, pair, latin_lid_fn) for w in words])
        t_ids, t_sw, t_dur, t_lid = align_to_subwords(words, lids, *make_labels(lids))
        if ids:
            ids.append(tokenizer.sep_token_id); sw.append(-100); dur.append(-1); lid.append(None)
        ids += t_ids; sw += t_sw; dur += t_dur; lid += t_lid
    if not any(s != -100 for s in sw):
        return None
    return {"ids": ids[:max_len], "sw": sw[:max_len], "dur": dur[:max_len], "lid": lid[:max_len],
            "pair": pair, "multi_turn": int(ex["conversation_type"] == "multi-turn")}

class StreamingCSDataset(IterableDataset):
    def __init__(self, hf_split, window=WINDOW, latin_lid_fn=LATIN_LID):
        self.data, self.window, self.latin_lid_fn = hf_split, window, latin_lid_fn

    def __iter__(self):
        info = get_worker_info()
        wid, nw = (info.id, info.num_workers) if info else (0, 1)
        for i, ex in enumerate(self.data):
            if i % nw != wid:
                continue
            out = process_example(ex, self.latin_lid_fn)
            if out is None:
                continue
            ids, sw, dur = out["ids"], out["sw"], out["dur"]
            for t in range(len(ids)):
                if sw[t] == -100:
                    continue
                start = max(0, t - self.window + 1)
                yield {"input_ids": ids[start:t + 1], "y_sw": sw[t], "y_dur": dur[t],
                       "pair_id": PAIR_ID[out["pair"]], "multi_turn": out["multi_turn"]}

def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    ids = [[pad] * (L - len(b["input_ids"])) + b["input_ids"] for b in batch]
    mask = [[0] * (L - len(b["input_ids"])) + [1] * len(b["input_ids"]) for b in batch]
    return {"input_ids": torch.tensor(ids), "attention_mask": torch.tensor(mask),
            **{k: torch.tensor([b[k] for b in batch]) for k in ["y_sw", "y_dur", "pair_id", "multi_turn"]}}

train_ds = StreamingCSDataset(splits["train"])
loader = DataLoader(train_ds, batch_size=32, collate_fn=collate)
batch = next(iter(loader))
{k: tuple(v.shape) for k, v in batch.items()}

{'input_ids': (32, 26),
 'attention_mask': (32, 26),
 'y_sw': (32,),
 'y_dur': (32,),
 'pair_id': (32,),
 'multi_turn': (32,)}

## 7. Label statistics (for the slides)
Switch events are rare, so report the class balance of `y_sw` and the distribution of `y_dur` bins per language pair.
All rows of each split are processed once here and reused by the baseline below.

In [16]:
processed = {k: [r for r in map(process_example, v) if r is not None] for k, v in splits.items()}

rows = []
for p in SELECTED_PAIRS:
    for multi in (0, 1):
        sw, dur = Counter(), Counter()
        rs = [r for r in processed["train"] if r["pair"] == p and r["multi_turn"] == multi]
        for r in rs:
            sw.update(s for s in r["sw"] if s != -100)
            dur.update(d for d in r["dur"] if d != -1)
        n = sum(sw.values()) or 1
        rows.append({"pair": "-".join(p), "conversation": "multi" if multi else "single", "rows": len(rs),
                     "positions": n, "switch %": round(100 * sw[1] / n, 2),
                     "Small": dur[0], "Medium": dur[1], "Large": dur[2]})
pd.DataFrame(rows)

,pair,conversation,rows,positions,switch %,Small,Medium,Large
0,en-ar,single,3199,100684,12.78,4223,3375,5274
1,en-ar,multi,3199,170889,8.68,5742,3656,5430
2,en-hi,single,1412,62072,19.45,6237,2788,3049
3,en-hi,multi,1304,87033,15.66,7831,2998,2797
4,en-yue,single,3199,312076,9.00,7847,5599,14629
5,en-yue,multi,3198,381286,5.65,10016,5253,6281
6,en-es,single,3199,116795,10.51,3625,2918,5728
7,en-es,multi,3199,219901,7.31,5246,4083,6743
8,en-ko,single,3199,102246,14.30,4805,3989,5824
9,en-ko,multi,3198,166952,9.84,5871,3662,6892


## 8. Naive baseline: predict the switch from the current token only
For each sub-word id at a labeled position, estimate `P(switch | x_t)` on **train** (add-one smoothed toward the global switch rate).
Predict a switch when the probability exceeds a threshold tuned for F1 on **val**; unseen tokens get the global rate.

Also shown: the trivial **always-switch** baseline (F1 = 2p/(1+p) for switch rate p). "Never switch" has F1 = 0.

In [17]:
def positions(rows):
    recs = [(PAIR_NAMES[PAIR_ID[r["pair"]]], i, s)
            for r in rows for i, s in zip(r["ids"], r["sw"]) if s != -100]
    return pd.DataFrame(recs, columns=["pair", "tok", "y_sw"])

P = {k: positions(v) for k, v in processed.items()}

tr = P["train"]
prior = tr.y_sw.mean()
g = tr.groupby("tok").y_sw.agg(["sum", "count"])
p_switch = ((g["sum"] + prior) / (g["count"] + 1)).to_dict()

for k in P:
    P[k]["p"] = P[k].tok.map(p_switch).fillna(prior)

grid = np.linspace(0.05, 0.95, 91)
thr = max(grid, key=lambda th: f1(P["val"].y_sw, (P["val"].p > th).astype(int)))
for k in P:
    P[k]["yhat"] = (P[k].p > thr).astype(int)
print(f"global switch rate (train) = {prior:.3f}, tuned threshold (val) = {thr:.2f}")

global switch rate (train) = 0.093, tuned threshold (val) = 0.32


In [18]:
te = P["test"]
res = []
for name, d in te.groupby("pair", sort=False):
    res.append({"pair": name, "positions": len(d), "switch %": round(100 * d.y_sw.mean(), 2),
                "F1 always-switch": round(f1(d.y_sw, np.ones(len(d), dtype=int)), 3),
                "F1 token-lookup": round(f1(d.y_sw, d.yhat), 3)})
pd.DataFrame(res).set_index("pair").reindex(PAIR_NAMES)

,positions,switch %,F1 always-switch,F1 token-lookup
pair,,,,
en-ar,33981,10.10,0.184,0.689
en-hi,18253,17.02,0.291,0.520
en-yue,88209,7.13,0.133,0.404
en-es,40675,8.17,0.151,0.647
en-ko,33727,11.47,0.206,0.752
en-de,31357,8.30,0.153,0.650


## Notes for Next Steps
- **Multi-turn**: switches at speaker changes are currently ignored. They could become a separate "cross-turn" evaluation slice.
- **Romanized Hindi** (~58% of single-turn rows) is dropped for now; labeling it needs a Hinglish word-level LID.
  Rows that mix Devanagari and romanized Hindi are kept and may carry some wrong labels.
- **Han characters**: each character is one "word", so yue durations are counted in characters while en/es/de count words; burst lengths are not directly comparable across pairs.
  `is_split_into_words=True` also inserts a stray `▁` before many Han characters, which differs from how XLM-R tokenizes running text. Tokenizing the raw sentence with `offset_mapping` would fix this.
- **CJK punctuation** attaches to the following Latin word (`，I`), and windows do not start with `<s>`.
- **Latin-pair LID** (es/de) is heuristic. Its agreement with LinCE / lingua is reported in Section 4; if it is low, the LinCE model could replace it for en-es.